# Image Classification with Deep Learning

A beginner-friendly deep learning project: train an image classifier on **any Kaggle image dataset** arranged as one folder per class.

**Pipeline**
1. Dataset preparation (train / validation / test)
2. Model: MobileNetV2 (transfer learning) + data augmentation
3. Training and validation (two phases: head, then fine-tuning)
4. Performance evaluation (accuracy, classification report, confusion matrix)
5. Testing predictions on new images

**Tech stack:** Python, TensorFlow/Keras, NumPy, Matplotlib, Scikit-learn, Pillow

## 0. Install requirements (run once)
Skip this cell if the packages are already installed.

In [ ]:
%pip install tensorflow numpy matplotlib scikit-learn pillow

## 1. Imports

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import callbacks, layers, models

print("TensorFlow:", tf.__version__)

## 2. Configuration
**Change these paths to your downloaded Kaggle dataset.** Each folder must contain one sub-folder per class.

- Dataset with separate train and test folders (e.g. Intel Image Classification): set both.
- Only one folder: set `TEST_DIR = None` and the data is split automatically.

In [ ]:
TRAIN_DIR = "seg_train"      # folder with one sub-folder per class
TEST_DIR = "seg_test"        # or None to split automatically

IMG_SIZE = 160
BATCH_SIZE = 32
EPOCHS = 8                        # phase 1: base model frozen
FINE_TUNE_EPOCHS = 8              # phase 2: unfreeze top of base model (0 to skip)
VAL_SPLIT = 0.15
OUTPUT_DIR = "outputs"
SEED = 42

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".gif")
AUTOTUNE = tf.data.AUTOTUNE
os.makedirs(OUTPUT_DIR, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)

In [ ]:
import os
print(os.getcwd())
print(os.path.exists("seg_train"), os.path.exists("data/seg_train"))

## 3. Dataset preparation
`resolve_root` handles Kaggle zips that nest folders (like `seg_train/seg_train/<classes>`).

In [ ]:
def resolve_root(path):
    """Walk down through single-folder wrappers until we reach the folder holding the classes."""
    while True:
        entries = os.listdir(path)
        subs = [d for d in entries if os.path.isdir(os.path.join(path, d))]
        has_images = any(f.lower().endswith(IMG_EXT) for f in entries)
        if len(subs) == 1 and not has_images:
            inner = os.path.join(path, subs[0])
            if any(os.path.isdir(os.path.join(inner, d)) for d in os.listdir(inner)):
                path = inner
                continue
        return path


def load_datasets():
    size = (IMG_SIZE, IMG_SIZE)
    train_root = resolve_root(TRAIN_DIR)
    print("Train folder:", train_root)

    if TEST_DIR:
        train_ds = tf.keras.utils.image_dataset_from_directory(
            train_root, validation_split=VAL_SPLIT, subset="training",
            seed=SEED, image_size=size, batch_size=BATCH_SIZE)
        val_ds = tf.keras.utils.image_dataset_from_directory(
            train_root, validation_split=VAL_SPLIT, subset="validation",
            seed=SEED, image_size=size, batch_size=BATCH_SIZE, shuffle=False)
        class_names = train_ds.class_names
        test_root = resolve_root(TEST_DIR)
        print("Test folder: ", test_root)
        test_ds = tf.keras.utils.image_dataset_from_directory(
            test_root, image_size=size, batch_size=BATCH_SIZE, shuffle=False,
            class_names=class_names)
    else:
        hold = min(VAL_SPLIT * 2, 0.5)
        train_ds = tf.keras.utils.image_dataset_from_directory(
            train_root, validation_split=hold, subset="training",
            seed=SEED, image_size=size, batch_size=BATCH_SIZE)
        rest = tf.keras.utils.image_dataset_from_directory(
            train_root, validation_split=hold, subset="validation",
            seed=SEED, image_size=size, batch_size=BATCH_SIZE, shuffle=False)
        class_names = train_ds.class_names
        n = int(rest.cardinality().numpy())
        val_ds, test_ds = rest.take(n // 2), rest.skip(n // 2)

    print(f"Classes ({len(class_names)}):", class_names)
    return (train_ds.prefetch(AUTOTUNE), val_ds.prefetch(AUTOTUNE),
            test_ds.prefetch(AUTOTUNE), class_names)


train_ds, val_ds, test_ds, class_names = load_datasets()

### Look at some training images

In [ ]:
plt.figure(figsize=(10, 8))
images, labels = next(iter(train_ds))
for i in range(min(12, len(images))):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Build the model
MobileNetV2 pretrained on ImageNet. Augmentation and scaling are **inside** the model, so raw 0-255 pixels can be fed at prediction time.

In [ ]:
def build_model(num_classes):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
    base.trainable = False

    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.RandomFlip("horizontal")(inputs)      # augmentation: training only
    x = layers.RandomRotation(0.08)(x)
    x = layers.RandomZoom(0.1)(x)
    x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)   # MobileNetV2 expects [-1, 1]
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inputs, outputs), base


def compile_model(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])


model, base = build_model(len(class_names))
model.summary()

## 5. Training and validation
**Phase 1** trains only the new classification head. **Phase 2** fine-tunes the top layers of MobileNetV2 with a very small learning rate.

In [ ]:
best_path = os.path.join(OUTPUT_DIR, "best_model.keras")
cbs = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    callbacks.ModelCheckpoint(best_path, monitor="val_accuracy", save_best_only=True),
]

# Phase 1: train the head
compile_model(model, 1e-3)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cbs)
hist = {k: list(v) for k, v in h1.history.items()}
split_at = len(hist["loss"])

In [ ]:
# Phase 2: fine-tune the top of the base model
if FINE_TUNE_EPOCHS > 0:
    base.trainable = True
    for layer in base.layers[:-40]:
        layer.trainable = False
    compile_model(model, 1e-5)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, callbacks=cbs)
    for k, v in h2.history.items():
        hist[k] += list(v)
else:
    split_at = None

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, key, title in zip(axes, ["accuracy", "loss"], ["Accuracy", "Loss"]):
    ax.plot(hist[key], label="train")
    ax.plot(hist["val_" + key], label="validation")
    if split_at:
        ax.axvline(split_at - 1, color="gray", linestyle="--", label="fine-tuning starts")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()

## 6. Performance evaluation on the test set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc:.4f}  |  Test loss: {test_loss:.4f}\n")

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = model.predict(test_ds, verbose=0).argmax(axis=1)

report = classification_report(y_true, y_pred, target_names=class_names, digits=3)
print(report)
with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w") as f:
    f.write(f"Test accuracy: {test_acc:.4f}\nTest loss: {test_loss:.4f}\n\n{report}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
n = len(class_names)
fig, ax = plt.subplots(figsize=(max(6, n * 0.7), max(5, n * 0.6)))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix (test set)")
if n <= 20:
    thresh = cm.max() / 2
    for i in range(n):
        for j in range(n):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] > thresh else "black")
fig.colorbar(im)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

### Sample predictions (green = correct, red = wrong)

In [ ]:
images, labels = next(iter(test_ds.unbatch().shuffle(1000, seed=SEED).batch(12)))
probs = model.predict(images, verbose=0)
preds = probs.argmax(axis=1)

plt.figure(figsize=(10, 8))
for i in range(len(images)):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(f"{class_names[preds[i]]} ({probs[i][preds[i]]:.0%})\ntrue: {class_names[labels[i]]}",
              color="green" if preds[i] == labels[i] else "red", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_predictions.png"), dpi=150)
plt.show()

## 7. Save the model

In [ ]:
model.save(os.path.join(OUTPUT_DIR, "final_model.keras"))
with open(os.path.join(OUTPUT_DIR, "meta.json"), "w") as f:
    json.dump({"class_names": class_names, "img_size": IMG_SIZE}, f)
print(f"Saved to '{OUTPUT_DIR}/'")

## 8. Test the model on a new image
Put any image path below and run the cell.

In [ ]:
from PIL import Image


def predict_image(path, top=3):
    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    batch = np.expand_dims(np.array(img, dtype="float32"), axis=0)
    probs = model.predict(batch, verbose=0)[0]
    best = np.argsort(probs)[::-1][:top]

    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{class_names[best[0]]} ({probs[best[0]]:.1%})")
    plt.show()
    for rank, i in enumerate(best, 1):
        print(f"{rank}. {class_names[i]:<16} {probs[i]:.1%}")


predict_image("my_image.jpg")   # <- change this path

## Ideas for improvement
- Try other backbones (EfficientNet, ResNet50) and compare.
- Rebuild the same pipeline in PyTorch.
- Add a small Streamlit/Gradio demo for uploading images.